# 01 · Renombrado de fotos y sorteo de conjuntos

Prepara el dataset de comprobantes.

1. Renombra las fotos de cada proveedor con un código (`provXX_001.jpg`, `provXX_002.jpg`...), para que los nombres de archivo no expongan datos de terceros.
2. Sortea con semilla fija 3 fotos de desarrollo por cada proveedor fijo. El resto, y todas las de proveedores ocasionales, quedan para prueba.
3. Carga la lista de archivos y su conjunto en la planilla de referencia.

Resultado obtenido: 53 fotos, 15 de desarrollo y 38 de prueba.

Las fotos y la planilla están en una carpeta privada de Drive y no se suben al repositorio.

## Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Renombrar y sortear

In [ ]:
import os, random, re

BASE = "/content/drive/MyDrive/5to cuatrimestre/Practica profesionalizante 3/PP3-boletas"
FOTOS = os.path.join(BASE, "fotos")

# Los nombres reales de cada código están en la pestaña "proveedores" de la planilla privada.
# Acá se usan nombres genéricos para no publicar datos de terceros.
PROVEEDORES = {
    "prov01": "Proveedor fijo 1",
    "prov02": "Proveedor fijo 2",
    "prov03": "Proveedor fijo 3",
    "prov04": "Proveedor fijo 4",
    "prov05": "Proveedor fijo 5",
    "prov06": "Proveedor ocasional 1",
    "prov07": "Proveedor ocasional 2",
    "prov08": "Proveedor ocasional 3",
    "prov09": "Proveedor ocasional 4",
}
FIJOS = ["prov01", "prov02", "prov03", "prov04", "prov05"]
N_DESARROLLO = 3   # fotos de desarrollo por proveedor fijo
SEMILLA = 42       # fija el sorteo para poder repetirlo

EXT = (".jpg", ".jpeg")
filas = []
rng = random.Random(SEMILLA)

for codigo in sorted(PROVEEDORES):
    carpeta = os.path.join(FOTOS, codigo)
    archivos = sorted(f for f in os.listdir(carpeta) if f.lower().endswith(EXT))
    ya_renombrados = all(re.fullmatch(rf"{codigo}_\d{{3}}\.jpg", f) for f in archivos)

    # 1) Renombrar a provXX_001.jpg, provXX_002.jpg... (solo la primera vez)
    if not ya_renombrados:
        nuevos = []
        for i, viejo in enumerate(archivos, start=1):
            nuevo = f"{codigo}_{i:03d}.jpg"
            os.rename(os.path.join(carpeta, viejo), os.path.join(carpeta, nuevo))
            nuevos.append(nuevo)
        archivos = nuevos

    # 2) Sorteo: 3 de desarrollo en los fijos; el resto y los ocasionales, prueba
    if codigo in FIJOS:
        desarrollo = set(rng.sample(archivos, min(N_DESARROLLO, len(archivos))))
    else:
        desarrollo = set()

    for f in archivos:
        conjunto = "desarrollo" if f in desarrollo else "prueba"
        filas.append([f, PROVEEDORES[codigo], conjunto])
    print(f"{codigo} {PROVEEDORES[codigo]:<17} {len(archivos):>2} fotos | desarrollo: {sorted(desarrollo)}")

n_dev = sum(1 for r in filas if r[2] == "desarrollo")
print(f"\nTotal: {len(filas)} fotos | desarrollo: {n_dev} | prueba: {len(filas) - n_dev}")

## Cargar la lista en la planilla de referencia

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

ID_PLANILLA = "ID_DE_LA_PLANILLA_PRIVADA"  # no se publica

planilla = gc.open_by_key(ID_PLANILLA)
hoja = planilla.worksheet("boletas")

if len(hoja.get_all_values()) > 1:
    print("La pestaña boletas ya tiene filas cargadas, no escribo nada.")
else:
    hoja.update(range_name="A2", values=[[f, p] for f, p, _ in filas])
    hoja.update(range_name="K2", values=[[c] for _, _, c in filas])
    print(f"Listo: {len(filas)} filas cargadas en la pestaña boletas.")